In [1]:
import hail as hl

In [2]:
hl.init(
spark_conf={
    'spark.hadoop.fs.gs.requester.pays.mode': 'CUSTOM',
    'spark.hadoop.fs.gs.requester.pays.buckets': 'regional_missense_constraint,gnomad-public-requester-pays,gnomad',
    'spark.hadoop.fs.gs.requester.pays.project.id': 'lily-sandbox-a29d'
}, default_reference='GRCh38'
)

/opt/conda/miniconda3/lib/python3.10/site-packages/hailtop/aiocloud/aiogoogle/user_config.py:43: UserWarning: Reading spark-defaults.conf to determine GCS requester pays configuration. This is deprecated. Please use `hailctl config set gcs_requester_pays/project` and `hailctl config set gcs_requester_pays/buckets`.
  warnings.warn(
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


SPARKMONITOR_LISTENER: Started SparkListener for Jupyter Notebook
SPARKMONITOR_LISTENER: Port obtained from environment: 39465
SPARKMONITOR_LISTENER: Application Started: application_1768867195754_0008 ...Start Time: 1768880044692


Running on Apache Spark version 3.3.0
SparkUI available at http://lw-m.us-central1-b.c.lily-sandbox-a29d.internal:41735
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.120-f00f916faf78
LOGGING: writing to /home/hail/hail-20260120-0333-0.2.120-f00f916faf78.log


In [3]:
import logging
import pickle
from itertools import combinations
from typing import Dict, List, Set, Tuple, Union
import numpy as np
import pandas as pd
from gnomad.resources.resource_utils import DataException
from gnomad.utils.file_utils import file_exists

In [4]:
from rmc.resources.basics import (    
    MPC_PREFIX,
    TEMP_PATH_WITH_FAST_DEL,
    TEMP_PATH_WITH_SLOW_DEL,
)
from rmc.resources.gnomad import constraint_ht
from rmc.resources.reference_data import (
    FOLD_K,
    blosum,
    blosum_txt_path,
    cadd,
    clinvar_plp_mis_haplo,
    grantham,
    grantham_txt_path,
    train_val_test_transcripts_path,
)
from rmc.resources.resource_utils import CURRENT_GNOMAD_VERSION
from rmc.resources.rmc import (
    CURRENT_FREEZE,
    amino_acids_oe_path,
    context_with_oe,
    context_with_oe_dedup,
    filtered_context,
    gnomad_fitted_score_path,
    joint_clinvar_gnomad_path,
    misbad_path,
    mpc_model_pkl_path,
    mpc_release,
)
from rmc.utils.generic import get_aa_map, get_gnomad_public_release, keep_criteria
from rmc.utils.missense_badness import variant_csq_expr
from rmc.utils.mpc import (
    convert_score_list_to_ht,
    import_blosum,
    import_grantham,
    prepare_pop_path_ht,
)

In [5]:
logging.basicConfig(
    format="%(asctime)s (%(name)s %(lineno)s): %(message)s",
    datefmt="%m/%d/%Y %I:%M:%S %p",
)
logger = logging.getLogger("calculate_aa_oe_metrics")
logger.setLevel(logging.INFO)

# Create HT of obs and exp counts by AA substitution type and missense/LoF OE bin

Assumes `prepare_amino_acid_ht()` has been run in `missense_badness.py` to label context HT variants in training transcripts with AA substitution type and OE.

In [6]:
constraint_ht.path

'gs://gnomad/v4.1/constraint_coverage_corrected/metrics/transcript_consequences/gnomad.v4.1.constraint_metrics.coverage_corrected.ht'

In [7]:
lof_ht = constraint_ht.ht()
lof_ht = lof_ht.key_by("transcript")

In [8]:
lof_ht.describe()

----------------------------------------
Global fields:
    'calculate_mu_globals': struct {
        freq_meta: array<dict<str, str>>, 
        ac_cutoff: int32, 
        min_cov: int32, 
        max_cov: int32, 
        gerp_lower_cutoff: float64, 
        gerp_upper_cutoff: float64, 
        genetic_ancestry_groups: array<str>, 
        downsampling_level: int32, 
        downsampling_idx: int32, 
        most_severe_consequence: array<str>
    } 
    'build_models_globals': struct {
        synonymous_transcript_filter_field: str, 
        low_cov_cutoff: int32, 
        high_cov_cutoff: int32, 
        upper_cov_cutoff: int32, 
        skip_coverage_model: bool
    } 
    'apply_models_globals': struct {
        low_cov_cutoff: int32, 
        high_cov_cutoff: int32, 
        skip_coverage_model: bool, 
        plateau_models: dict<struct {
            cpg: bool, 
            genomic_region: str
        }, array<array<float64>>>, 
        coverage_model: array<float64>, 
        lo

In [9]:
lof_ht.show()

,,,,,,,,,,,,,,,
gene,gene_id,transcript,canonical,mane_select,constraint_groups,no_variants,constraint_flags,pLI,pNull,pRec,level,transcript_type,chromosome,cds_length,num_coding_exons
str,str,str,bool,bool,"array<struct{mu_snp: float64, mu: float64, possible_variants: int64, coverage_correction: float64, oe_info: array<struct{observed_variants: int64, predicted_proportion_observed: float64, expected_variants: float64, oe: float64, oe_ci: struct{lower: float64, upper: float64}, z_raw: float64}>, flags: set<str>, z_score: float64, upper_rank: int64, upper_bin_sextile: int32, upper_bin_decile: int32}>",bool,set<str>,float64,float64,float64,str,str,str,int64,int64
"""ARF5""","""ENSG00000004059""","""ENST00000000233""",True,True,"[(2.66e-06,2.69e-06,359,3.64e+02,[(102,9.37e+01,9.51e+01,1.07e+00,(9.13e-01,1.26e+00),-7.07e-01),(8,1.26e+01,1.28e+01,6.25e-01,(3.66e-01,1.13e+00),1.34e+00),(10,1.27e+01,1.28e+01,7.80e-01,(4.80e-01,1.32e+00),7.89e-01),(8,1.09e+01,1.10e+01,7.25e-01,(4.25e-01,1.31e+00),9.13e-01),(75,6.46e+01,6.55e+01,1.15e+00,(9.49e-01,1.39e+00),-1.17e+00),(22,2.05e+01,2.07e+01,1.06e+00,(7.57e-01,1.51e+00),-2.77e-01)],{},-4.22e-01,15851,3,6),(6.77e-06,6.85e-06,1172,1.19e+03,[(135,2.36e+02,2.39e+02,5.64e-01,(4.90e-01,6.51e-01),6.74e+00),(8,3.17e+01,3.21e+01,2.49e-01,(1.45e-01,4.50e-01),4.25e+00),(10,3.18e+01,3.22e+01,3.11e-01,(1.91e-01,5.27e-01),3.91e+00),(14,2.75e+01,2.78e+01,5.04e-01,(3.32e-01,7.87e-01),2.62e+00),(102,1.61e+02,1.63e+02,6.25e-01,(5.31e-01,7.36e-01),4.80e+00),(26,5.13e+01,5.19e+01,5.01e-01,(3.66e-01,6.95e-01),3.60e+00)],{},2.87e+00,1214,0,0),(4.71e-07,4.77e-07,120,1.21e+02,[(10,2.45e+01,2.48e+01,4.03e-01,(2.48e-01,6.84e-01),2.97e+00),(0,2.57e+00,2.60e+00,0.00e+00,(0.00e+00,1.11e+00),1.61e+00),(0,2.56e+00,2.59e+00,0.00e+00,(0.00e+00,1.12e+00),1.61e+00),(2,2.18e+00,2.21e+00,9.06e-01,(3.40e-01,1.86e+00),1.40e-01),(6,1.60e+01,1.62e+01,3.69e-01,(2.01e-01,7.29e-01),2.54e+00),(1,4.35e+00,4.41e+00,2.27e-01,(8.00e-02,1.07e+00),1.62e+00)],{},2.16e+00,5926,1,2),(4.71e-07,4.77e-07,120,1.21e+02,[(10,2.45e+01,2.48e+01,4.03e-01,(2.48e-01,6.84e-01),2.97e+00),(0,2.57e+00,2.60e+00,0.00e+00,(0.00e+00,1.11e+00),1.61e+00),(0,2.56e+00,2.59e+00,0.00e+00,(0.00e+00,1.12e+00),1.61e+00),(2,2.18e+00,2.21e+00,9.06e-01,(3.40e-01,1.86e+00),1.40e-01),(6,1.60e+01,1.62e+01,3.69e-01,(2.01e-01,7.29e-01),2.54e+00),(1,4.35e+00,4.41e+00,2.27e-01,(8.00e-02,1.07e+00),1.62e+00)],{},2.48e+00,5802,1,2),(4.71e-07,4.77e-07,120,1.21e+02,[(10,2.45e+01,2.48e+01,4.03e-01,(2.48e-01,6.84e-01),2.97e+00),(0,2.57e+00,2.60e+00,0.00e+00,(0.00e+00,1.11e+00),1.61e+00),(0,2.56e+00,2.59e+00,0.00e+00,(0.00e+00,1.12e+00),1.61e+00),(2,2.18e+00,2.21e+00,9.06e-01,(3.40e-01,1.86e+00),1.40e-01),(6,1.60e+01,1.62e+01,3.69e-01,(2.01e-01,7.29e-01),2.54e+00),(1,4.35e+00,4.41e+00,2.27e-01,(8.00e-02,1.07e+00),1.62e+00)],{},2.48e+00,5802,1,2),(4.71e-07,4.77e-07,120,1.21e+02,[(10,2.45e+01,2.48e+01,4.03e-01,(2.48e-01,6.84e-01),2.97e+00),(0,2.57e+00,2.60e+00,0.00e+00,(0.00e+00,1.11e+00),1.61e+00),(0,2.56e+00,2.59e+00,0.00e+00,(0.00e+00,1.12e+00),1.61e+00),(2,2.18e+00,2.21e+00,9.06e-01,(3.40e-01,1.86e+00),1.40e-01),(6,1.60e+01,1.62e+01,3.69e-01,(2.01e-01,7.29e-01),2.54e+00),(1,4.35e+00,4.41e+00,2.27e-01,(8.00e-02,1.07e+00),1.62e+00)],{},2.52e+00,5699,1,2)]",False,{},3.64e-01,3.79e-03,6.32e-01,"""2""","""protein_coding""","""chr7""",540,6
"""M6PR""","""ENSG00000003056""","""ENST00000000412""",True,True,"[(2.76e-06,2.76e-06,543,5.43e+02,[(111,1.23e+02,1.23e+02,8.99e-01,(7.70e-01,1.05e+00),1.12e+00),(13,1.43e+01,1.43e+01,9.10e-01,(5.92e-01,1.45e+00),3.39e-01),(15,1.43e+01,1.43e+01,1.05e+00,(7.02e-01,1.61e+00),-1.95e-01),(10,1.22e+01,1.22e+01,8.19e-01,(5.04e-01,1.39e+00),6.32e-01),(73,8.23e+01,8.23e+01,8.87e-01,(7.33e-01,1.08e+00),1.03e+00),(17,2.38e+01,2.38e+01,7.16e-01,(4.89e-01,1.07e+00),1.39e+00)],{},6.67e-01,4689,1,1),(8.33e-06,8.33e-06,1811,1.81e+03,[(295,3.36e+02,3.36e+02,8.78e-01,(7.98e-01,9.67e-01),2.24e+00),(30,4.10e+01,4.10e+01,7.31e-01,(5.46e-01,9.92e-01),1.72e+00),(26,4.11e+01,

In [10]:
def reformat_constraint_ht(
    ht, 
    constraint_groups_to_keep = {"csq_set": ['syn', 'mis'], "lof": ['hc']}, 
    fields_to_keep = ["observed_variants", "expected_variants"],
    filter_to_enst=True, 
    filter_to_canonical=True, 
    include_pops=True,
):
    constraint_group_meta = hl.eval(ht.constraint_group_meta)
    print("constraint_group_meta before re-naming", constraint_group_meta)
    constraint_group_meta = [
        '_'.join([v if k == "csq_set" else f"{k}_{v}" for k, v in m.items()])
        if any([(k, g) in list(m.items()) for k, v in constraint_groups_to_keep.items() for g in v]) else None
        for m in constraint_group_meta
    ]
    print("constraint_group_meta after re-naming", constraint_group_meta)
    
    if filter_to_enst or filter_to_canonical:
        filter_expr = True
        if filter_to_enst:
            filter_expr &= ht.transcript.startswith("ENST")
        if filter_to_canonical:
            filter_expr &= ht.canonical

        ht = ht.filter(filter_expr)
        
        
    ann_expr = {}
    for i, m in enumerate(constraint_group_meta):
        if m:
            g = ht.constraint_groups[i]
            ann_expr[m] = g.select(*[f for f in fields_to_keep if f in g]).annotate(
                **g.oe_info[0].select(*[f for f in fields_to_keep if f in g.oe_info[0]])
            )
            if include_pops:
                pop_expr = {
                    pop_m['gen_anc']: g.oe_info[j].select(*[f for f in fields_to_keep if f in g.oe_info[j]])
                    for j, pop_m in enumerate(hl.eval(ht.exomes_freq_meta))
                    if "gen_anc" in pop_m and len(pop_m) == 2
                }
                ann_expr[m] = ann_expr[m].annotate(**pop_expr)

    ht = ht.annotate(**ann_expr).drop("constraint_groups", "no_variants")

    return ht

In [11]:
lof_ht = reformat_constraint_ht(lof_ht)

constraint_group_meta before re-naming [{'csq_set': 'syn'}, {'csq_set': 'mis'}, {'lof': 'classic'}, {'lof': 'hc_lc'}, {'lof': 'classic_hc_lc'}, {'lof': 'hc'}]
constraint_group_meta after re-naming ['syn', 'mis', None, None, None, 'lof_hc']


In [12]:
lof_ht.describe()

----------------------------------------
Global fields:
    'calculate_mu_globals': struct {
        freq_meta: array<dict<str, str>>, 
        ac_cutoff: int32, 
        min_cov: int32, 
        max_cov: int32, 
        gerp_lower_cutoff: float64, 
        gerp_upper_cutoff: float64, 
        genetic_ancestry_groups: array<str>, 
        downsampling_level: int32, 
        downsampling_idx: int32, 
        most_severe_consequence: array<str>
    } 
    'build_models_globals': struct {
        synonymous_transcript_filter_field: str, 
        low_cov_cutoff: int32, 
        high_cov_cutoff: int32, 
        upper_cov_cutoff: int32, 
        skip_coverage_model: bool
    } 
    'apply_models_globals': struct {
        low_cov_cutoff: int32, 
        high_cov_cutoff: int32, 
        skip_coverage_model: bool, 
        plateau_models: dict<struct {
            cpg: bool, 
            genomic_region: str
        }, array<array<float64>>>, 
        coverage_model: array<float64>, 
        lo

In [13]:
def aggregate_aa_and_filter_oe(
    ht: hl.Table,
    keep_non: bool,
    oe_col: str,
    min_oe: float = None,
    max_oe: float = None,
) -> hl.Table:
    # NOTE: `keep_non` exists so obs and exp values can be computed for nonsense substitutions
    # separately from missense, synonymous, and read-through substitutions
    logger.info("Filtering HT on mutation types and OE values...")
    mut_type_filter_expr = (ht.mut_type == "non") if keep_non else (ht.mut_type != "non")    
    oe_filter_expr = True
    if min_oe:
        oe_filter_expr = oe_filter_expr & (ht[oe_col] > min_oe)
    if max_oe:
        oe_filter_expr = oe_filter_expr & (ht[oe_col] <= max_oe)
    ht = ht.filter(mut_type_filter_expr & oe_filter_expr)
    
    if keep_non:
        # Filter LoF variants to those in genes with >= 16 expected LoF
        min_exp_lof_transcript_ht = lof_ht.filter(lof_ht.lof_hc.expected_variants >= 16)
        min_exp_lof_transcripts = hl.literal(
            min_exp_lof_transcript_ht.aggregate(
                hl.agg.collect_as_set(min_exp_lof_transcript_ht.transcript)
            )
        )
        ht = ht.filter(min_exp_lof_transcripts.contains(ht.transcript))

    logger.info("Grouping HT and aggregating observed and expected variant counts...")
    return ht.group_by("ref", "alt").aggregate(
        obs=hl.agg.sum(ht.observed),
        exp=hl.agg.sum(ht.expected),
        mut_type=hl.agg.take(ht.mut_type, 1)[0],
    )

In [14]:
from rmc.utils.constraint import get_constraint_transcripts

In [15]:
pass_transcripts = hl.eval(get_constraint_transcripts(filter_to_canonical=True, outlier=False))

WARNING (regional_missense_constraint_generic 630): Assumes LoF constraint has been separately calculated and that constraint HT exists...


In [16]:
len(pass_transcripts)

17841

In [17]:
context_ht = hl.read_table(
    "gs://gnomad/v4.1/constraint_coverage_corrected/preprocessed_data/gnomad.v4.1.context.preprocessed.ht"
).select_globals()

In [18]:
context_ht.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'context': str 
    'vep': struct {
        most_severe_consequence: str, 
        transcript_consequences: array<struct {
            transcript_id: str, 
            gene_id: str, 
            gene_symbol: str, 
            biotype: str, 
            most_severe_consequence: str, 
            mane_select: str, 
            canonical: int32, 
            lof: str, 
            lof_flags: str, 
            sift_score: float64, 
            polyphen_score: float64, 
            domains: array<struct {
                db: str, 
                name: str
            }>, 
            uniprot_isoform: array<str>, 
            amino_acids: str, 
            codons: str
        }>
    } 
    'ref': str 
    'alt': str 
    'was_flipped': bool 
    'transition': bool 
    'cpg': bool 
    'mutation_type': str 
    'muta

In [19]:
context_ht = context_ht.select(transcript_consequences=context_ht.vep.transcript_consequences)

In [20]:
# Filter to loci in canonical constraint pass transcripts only
# and filter transcript consequences to those transcripts too
pass_context_ht = context_ht.annotate(
    transcript_consequences=context_ht.transcript_consequences.filter(
        lambda x: hl.literal(pass_transcripts).contains(x.transcript_id)
    )
)
pass_context_ht = pass_context_ht.filter(
    hl.len(pass_context_ht.transcript_consequences) > 0
)

In [21]:
pass_context_ht = pass_context_ht.checkpoint(
    f"{TEMP_PATH_WITH_SLOW_DEL}/mpc/pass_preprocessed_context.ht",
    _read_if_exists=True,
    overwrite=False,
)

In [22]:
def filter_vars_in_transcripts_exclusive(ht, keep_transcripts_expr):
    # Filter variant-level table
    # NOTE: Function re-keys `ht` to locus and alleles only (removes transcript from key)
    ht = ht.key_by("locus", "alleles")
    # Filter to retain variants only in transcripts to keep
    # and not in any other transcripts including overlaps
    # This means that if there is a variant in the training transcripts and the test transcripts
    # (which are mutually exclusive), that variant will never be output from this function
    ht = ht.join(
        pass_context_ht.filter(
            pass_context_ht.transcript_consequences.all(
                lambda x: keep_transcripts_expr.contains(x.transcript_id)
            )
        ).select()
    )
    # Filter to remove variants in any other transcripts including overlaps
    ht = ht.anti_join(
        pass_context_ht.filter(
            pass_context_ht.transcript_consequences.any(
                lambda x: ~keep_transcripts_expr.contains(x.transcript_id)
            )
        ).select()
    )
    return ht

In [23]:
# Filter to high coverage regions
def filt_ht_by_mc_region_median_an(ht, an_threshold = 90):
    median_an_all_mc_intervals_ht = hl.read_table(
        "gs://regional_missense_constraint/temp/all_rmc_intervals_median_AN.ht"
    )
    high_cov_median_an_all_mc_intervals_ht = median_an_all_mc_intervals_ht.filter(
        median_an_all_mc_intervals_ht.median_exomes_AN_percent >= an_threshold
    )
    # Explode to match on both locus and transcript
    expl_high_cov_median_an_all_mc_intervals_ht = explode_intervals_to_loci(
        high_cov_median_an_all_mc_intervals_ht,
        interval_field="interval", keep_intervals=True,
    ).key_by("locus", "transcript")
    # Filter to high coverage regions with median AN>=90% over loci used in RMC calculation
    ht = ht.filter(
        hl.is_defined(
            expl_high_cov_median_an_all_mc_intervals_ht[ht.locus, ht.transcript]
        )
    )
    return ht

In [24]:
def amino_acids_oe_path(
    is_train: bool = False,
    freeze: int = CURRENT_FREEZE,
) -> str:
    """
    Return path to Table containing all possible amino acid substitutions and their missense OE ratio.

    Table is input to missense badness calculations.

    :param int freeze: RMC data freeze number. Default is CURRENT_FREEZE.
    :return: Path to Table.
    """
    return f"{MPC_PREFIX}/{CURRENT_GNOMAD_VERSION}/{freeze}/{'train/' if is_train else ''}amino_acid_oe.ht"

In [25]:
amino_acids_oe_path(is_train=False, freeze=CURRENT_FREEZE)

'gs://regional_missense_constraint/MPC/4.1/2/amino_acid_oe.ht'

In [26]:
hl.read_table(amino_acids_oe_path(is_train=False, freeze=CURRENT_FREEZE)).show()

,,,,,,,,,
locus,alleles,transcript,ref,alt,observed,expected,codons,amino_acids,oe
locus<GRCh38>,array<str>,str,str,str,int32,float64,str,str,float64
chr1:944697,"[""G"",""C""]","""ENST00000327044""","""Asp""","""Glu""",0,1.65e-01,"""gaC/gaG""","""D/E""",1.07e+00
chr1:944698,"[""T"",""A""]","""ENST00000327044""","""Asp""","""Val""",0,1.15e-01,"""gAc/gTc""","""D/V""",1.07e+00
chr1:944698,"[""T"",""G""]","""ENST00000327044""","""Asp""","""Ala""",0,8.01e-02,"""gAc/gCc""","""D/A""",1.07e+00
chr1:944699,"[""C"",""G""]","""ENST00000327044""","""Asp""","""His""",0,2.78e-01,"""Gac/Cac""","""D/H""",1.07e+00
chr1:944699,"[""C"",""T""]","""ENST00000327044""","""Asp""","""Asn""",1,8.17e-01,"""Gac/Aac""","""D/N""",1.07e+00
chr1:944700,"[""G"",""A""]","""ENST00000327044""","""Asp""","""Asp""",1,9.13e-01,"""gaC/gaT""","""D""",1.07e+00
chr1:944700,"[""G"",""C""]","""ENST00000327044""","""Asp""","""Glu""",1,2.49e-01,"""gaC/gaG""","""D/E""",1.07e+00
chr1:944700,"[""G"",""T""]","""ENST00000327044""","""Asp""","""Glu""",1,3.28e-01,"""gaC/gaA""","""D/E""",1.07e+00


In [27]:
def amino_acids_obs_exp_by_oe_bin_path(
    is_train: bool = False,
    freeze: int = CURRENT_FREEZE,
) -> str:
    """
    Table containing observed and expected counts per OE bin for each amino acid substitution type.

    # TODO: Add differences between LoF, missense, OE bin specification
    
    :param int freeze: RMC data freeze number. Default is CURRENT_FREEZE.
    :return: Path to Table.
    """
    return f"{MPC_PREFIX}/{CURRENT_GNOMAD_VERSION}/{freeze}/{'train/' if is_train else ''}amino_acid_obs_exp_by_oe_bin.ht"

In [30]:
def calculate_aa_obs_exp_over_oe_bins(
    overwrite_temp: bool,
    overwrite_output: bool,
    oe_bin_interval: float = 0.1,
    is_train: bool = False,
    freeze: int = CURRENT_FREEZE,
) -> None:
    # Read in coding context table annotated with codons and O/E
    # NOTE: This table is already filtered to regions/transcripts with AN>=60%
    # for the non-training table
    aa_oe_path = amino_acids_oe_path(is_train=is_train, freeze=freeze)
    if not file_exists(aa_oe_path):
        raise DataException(
            "Table with all amino acid substitutions and missense OE doesn't exist!"
        )
    logger.info("Reading in table with amino acid substitutions and missense OE...")
    ht = hl.read_table(aa_oe_path)
    
    logger.info("Adding variant consequence (mut_type) annotation...")
    ht = ht.annotate(
        mut_type=variant_csq_expr(ht.ref, ht.alt)
    )    
    logger.info("Adding LoF OE annotation...")
    ht = ht.rename({"oe": "missense_oe"})
    ht = ht.annotate(
        gene_lof_oe = (
            lof_ht[ht.transcript].lof_hc.observed_variants
            / lof_ht[ht.transcript].lof_hc.expected_variants
        )
    )
    ht = ht.checkpoint(
        f"{TEMP_PATH_WITH_FAST_DEL}/amino_acids_w_lof_oe.ht",
        _read_if_exists=not overwrite_temp,
        overwrite=overwrite_temp,
    )
    
    if is_train:
        logger.info("Filtering to transcripts in training set...")
        keep_transcripts = hl.experimental.read_expression(
            train_val_test_transcripts_path(fold=fold)
        )
        ht = filter_vars_in_transcripts_exclusive(ht, keep_transcripts)
        ht = ht.checkpoint(
            f"{TEMP_PATH_WITH_FAST_DEL}/amino_acids_w_lof_oe_train.ht",
            _read_if_exists=not overwrite_temp,
            overwrite=overwrite_temp,
        )

    logger.info(
        "Splitting input Table by OE to get observed/expected rates for each OE bin..."
    )
    # Create OE bin labels
    oe_bin_mins = hl.eval(hl.range(0, 10, int(oe_bin_interval*10))/10)
    oe_bin_min_labels = hl.eval([hl.format("%.1f", x) for x in oe_bin_mins])
    oe_bin_maxs = [x + oe_bin_interval for x in oe_bin_mins]
    oe_bin_max_labels = hl.eval([hl.format("%.1f", x) for x in oe_bin_maxs])

    hts_by_oe = {}
    for keep_non in [True, False]:
        oe_type = "gene_lof" if keep_non else "missense"
        hts_by_oe[oe_type] = {}
        # NOTE that hts_by_oe["gene_lof"] contains nonsense variants filtered based on gene LoF OE
        # and hts_by_oe["missense_oe"] contains all other variant types
        # filtered based on RMC missense OE

        for i, oe_bin_min in enumerate(oe_bin_mins):
            oe_bin_max = oe_bin_maxs[i]
            oe_bin_min_label = oe_bin_min_labels[i]
            oe_bin_max_label = oe_bin_max_labels[i]
            logger.info(
                "Creating HT for %s %s %s OE <= %s%s...",
                oe_bin_min_label,
                "<" if i != 0 else "<=",
                oe_type,
                oe_bin_max_label,
                "" if i != (len(oe_bin_mins) - 1) else "+",
            )
            ht_by_oe = aggregate_aa_and_filter_oe(
                ht,
                keep_non=keep_non,
                oe_col="gene_lof_oe" if keep_non else "missense_oe",
                min_oe=oe_bin_min if i != 0 else None,
                max_oe=oe_bin_max if i != (len(oe_bin_mins) - 1) else None,
            )
            ht_by_oe = ht_by_oe.annotate(
                obs_exp = ht_by_oe.obs / ht_by_oe.exp,
                oe_bin=hl.format("%s-%s", oe_bin_min_label, oe_bin_max_label),
            )
            hts_by_oe[oe_type][oe_bin_min_label] = ht_by_oe.checkpoint(
                f"{TEMP_PATH_WITH_SLOW_DEL}/amino_acids_{oe_bin_min_label}_{oe_bin_max_label}_{oe_type}_oe{'_train' if is_train else ''}.ht",
                _read_if_exists=not overwrite_temp,
                overwrite=overwrite_temp,
            )

    logger.info("Unioning HTs by constraint bin and mutation type...")
    # Initialize empty HT
    obs_exp_ht = hts_by_oe["missense"][oe_bin_min_labels[0]].head(0)
    # Union all HTs together
    for oe_type in hts_by_oe.keys():
        for i, oe_bin_min in enumerate(oe_bin_mins):
            oe_bin_min_label = oe_bin_min_labels[i]
            obs_exp_ht = obs_exp_ht.union(hts_by_oe[oe_type][oe_bin_min_label])
    obs_exp_ht = obs_exp_ht.naive_coalesce(10)
    obs_exp_ht.write(
        amino_acids_obs_exp_by_oe_bin_path(is_train=is_train, freeze=freeze),
        overwrite=overwrite_output,
    )

In [31]:
calculate_aa_obs_exp_over_oe_bins(
    overwrite_temp=True,
    overwrite_output=True,
)

INFO (calculate_aa_oe_metrics 16): Reading in table with amino acid substitutions and missense OE...
INFO (calculate_aa_oe_metrics 22): Adding variant consequence (mut_type) annotation...
INFO (calculate_aa_oe_metrics 26): Adding LoF OE annotation...
2026-01-20 04:18:19.863 Hail: INFO: Ordering unsorted dataset with network shuffle
2026-01-20 04:18:21.930 Hail: INFO: Ordering unsorted dataset with network shuffle
2026-01-20 04:19:12.943 Hail: INFO: Coerced sorted dataset==>(4339 + 26) / 4364]
2026-01-20 04:19:14.601 Hail: INFO: Ordering unsorted dataset with network shuffle
2026-01-20 04:19:48.709 Hail: INFO: Ordering unsorted dataset with network shuffle
2026-01-20 04:26:26.236 Hail: INFO: wrote table with 86414950 rows in 4364 partitions to gs://gnomad-tmp-4day/rmc/amino_acids_w_lof_oe.ht
INFO (calculate_aa_oe_metrics 52): Splitting input Table by OE to get observed/expected rates for each OE bin...
INFO (calculate_aa_oe_metrics 73): Creating HT for 0.0 <= gene_lof OE <= 0.1...
INFO 

INFO (calculate_aa_oe_metrics 10): Filtering HT on mutation types and OE values...
INFO (calculate_aa_oe_metrics 29): Grouping HT and aggregating observed and expected variant counts...
2026-01-20 04:30:40.226 Hail: INFO: Ordering unsorted dataset with network shuffle
2026-01-20 04:30:51.575 Hail: INFO: wrote table with 168 rows in 168 partitions to gs://gnomad-tmp/rmc/amino_acids_0.5_0.6_missense_oe.ht
INFO (calculate_aa_oe_metrics 73): Creating HT for 0.6 < missense OE <= 0.7...
INFO (calculate_aa_oe_metrics 10): Filtering HT on mutation types and OE values...
INFO (calculate_aa_oe_metrics 29): Grouping HT and aggregating observed and expected variant counts...
2026-01-20 04:30:57.820 Hail: INFO: Ordering unsorted dataset with network shuffle
Exception in thread "Thread-37" java.lang.NullPointerException3730 + 77) / 4364]
	at sparkmonitor.listener.JupyterSparkMonitorListener$TaskUpdaterThread.$anonfun$run$1(CustomListener.scala:116)
	at scala.collection.TraversableLike$grouper$1$.app

In [32]:
hl.read_table(amino_acids_obs_exp_by_oe_bin_path(is_train=False, freeze=CURRENT_FREEZE)).show()

,,,,,,
ref,alt,obs,exp,mut_type,obs_exp,oe_bin
str,str,int64,float64,str,float64,str
"""Ala""","""Ala""",687,8.14e+02,"""syn""",8.44e-01,"""0.0-0.1"""
"""Ala""","""Ala""",1350,1.47e+03,"""syn""",9.15e-01,"""0.1-0.2"""
"""Ala""","""Ala""",2728,2.71e+03,"""syn""",1.01e+00,"""0.2-0.3"""
"""Ala""","""Ala""",4966,4.82e+03,"""syn""",1.03e+00,"""0.3-0.4"""
"""Ala""","""Ala""",7871,7.79e+03,"""syn""",1.01e+00,"""0.4-0.5"""
"""Ala""","""Ala""",12608,1.23e+04,"""syn""",1.02e+00,"""0.5-0.6"""
"""Ala""","""Ala""",20973,2.06e+04,"""syn""",1.02e+00,"""0.6-0.7"""
"""Ala""","""Ala""",35655,3.52e+04,"""syn""",1.01e+00,"""0.7-0.8"""


In [43]:
hl.read_table(amino_acids_obs_exp_by_oe_bin_path(is_train=False, freeze=CURRENT_FREEZE)).show()

,,,,,,
ref,alt,obs,exp,mut_type,obs_exp,oe_bin
str,str,int64,float64,str,float64,str
"""Ala""","""Ala""",707,9.03e+02,"""syn""",7.83e-01,"""0.0-0.1"""
"""Ala""","""Ala""",1405,1.56e+03,"""syn""",9.01e-01,"""0.1-0.2"""
"""Ala""","""Ala""",2824,2.83e+03,"""syn""",9.99e-01,"""0.2-0.3"""
"""Ala""","""Ala""",5085,4.93e+03,"""syn""",1.03e+00,"""0.3-0.4"""
"""Ala""","""Ala""",8017,7.92e+03,"""syn""",1.01e+00,"""0.4-0.5"""
"""Ala""","""Ala""",12760,1.24e+04,"""syn""",1.03e+00,"""0.5-0.6"""
"""Ala""","""Ala""",21222,2.08e+04,"""syn""",1.02e+00,"""0.6-0.7"""
"""Ala""","""Ala""",36006,3.55e+04,"""syn""",1.01e+00,"""0.7-0.8"""


# Calculate overall OE and second derivative of OE over OE bins per AA substitution

In [33]:
amino_acids_obs_exp_by_oe_bin_ht = hl.read_table(
    amino_acids_obs_exp_by_oe_bin_path(is_train=False, freeze=CURRENT_FREEZE)
)

In [34]:
amino_acids_obs_exp_by_oe_bin_ht.filter(
    (amino_acids_obs_exp_by_oe_bin_ht.ref == "STOP")
).show()

,,,,,,
ref,alt,obs,exp,mut_type,obs_exp,oe_bin
str,str,int64,float64,str,float64,str
"""STOP""","""Arg""",0,3.49e-01,"""rdt""",0.00e+00,"""0.2-0.3"""
"""STOP""","""Arg""",0,3.49e-01,"""rdt""",0.00e+00,"""0.3-0.4"""
"""STOP""","""Arg""",0,2.76e-01,"""rdt""",0.00e+00,"""0.7-0.8"""
"""STOP""","""Arg""",1,1.29e+00,"""rdt""",7.76e-01,"""0.8-0.9"""
"""STOP""","""Arg""",10,1.06e+01,"""rdt""",9.45e-01,"""0.9-1.0"""
"""STOP""","""Cys""",0,9.96e-02,"""rdt""",0.00e+00,"""0.1-0.2"""
"""STOP""","""Cys""",0,1.95e-01,"""rdt""",0.00e+00,"""0.2-0.3"""
"""STOP""","""Cys""",0,1.73e-01,"""rdt""",0.00e+00,"""0.3-0.4"""


Looks like the readthrough coding variants aren't consistently retained in the new version of the gnomAD methods so we will just exclude these.

In [36]:
import numpy.polynomial.polynomial as poly

In [37]:
def calculate_aa_oe_overall(ht):
    return ht.group_by("ref", "alt").aggregate(
        aa_oe_overall=hl.agg.sum(ht.obs) / hl.agg.sum(ht.exp)
    )

def calculate_aa_oe_second_deriv(ht):
    # Convert to numpy array
    agg_ht = ht.select("obs_exp").collect_by_key()
    # Calculate second derivative of OE across bins
    agg_ht = agg_ht.annotate(x=hl.range(10))
    df = agg_ht.transmute(obs_exp=agg_ht.values.obs_exp).to_pandas()
    df["aa_oe_second_deriv"] = df.apply(lambda x: poly.polyfit(x["x"], x["obs_exp"], 2)[2], axis=1)
    return hl.Table.from_pandas(df).key_by("ref", "alt").select("aa_oe_second_deriv")

def calculate_aa_oe_metrics(ht: hl.Table):
    ht = ht.filter(ht.ref != "STOP")
    metrics_ht = calculate_aa_oe_overall(ht).join(calculate_aa_oe_second_deriv(ht))
    return metrics_ht

In [38]:
aa_oe_metrics_ht = calculate_aa_oe_metrics(
    hl.read_table(amino_acids_obs_exp_by_oe_bin_path(is_train=False, freeze=CURRENT_FREEZE))
)

In [39]:
aa_oe_metrics_ht = aa_oe_metrics_ht.naive_coalesce(1)

In [40]:
aa_oe_metrics_ht = aa_oe_metrics_ht.checkpoint(
    f"{MPC_PREFIX}/{CURRENT_GNOMAD_VERSION}/{CURRENT_FREEZE}/amino_acid_oe_metrics.ht",
    _read_if_exists=False,
    overwrite=True,
)

2026-01-20 04:33:09.251 Hail: INFO: Coerced sorted dataset
2026-01-20 04:33:13.461 Hail: INFO: wrote table with 178 rows in 1 partition to gs://regional_missense_constraint/MPC/4.1/2/amino_acid_oe_metrics.ht


In [41]:
aa_oe_metrics_ht.show()

,,,
ref,alt,aa_oe_overall,aa_oe_second_deriv
str,str,float64,float64
"""Ala""","""Ala""",1.08e+00,-2.35e-03
"""Ala""","""Asp""",8.86e-01,1.18e-02
"""Ala""","""Glu""",8.44e-01,8.09e-03
"""Ala""","""Gly""",9.56e-01,7.82e-04
"""Ala""","""Pro""",7.71e-01,9.01e-03
"""Ala""","""Ser""",1.10e+00,-1.32e-03
"""Ala""","""Thr""",1.02e+00,-2.50e-03
"""Ala""","""Val""",1.01e+00,-5.14e-04


In [42]:
aa_oe_metrics_ht.count()

178